## Detectron2 segmentation training

This notebook trains a Detectron2 Mask R-CNN model for plant segmentation.

Basic workflow:
1. Set project paths and imports.
2. Register COCO-format train/test datasets.
3. Train the segmentation model.
4. Evaluate and visualize predictions.

In [ ]:
import random
from pathlib import Path

import cv2
import detectron2
import matplotlib.pyplot as plt
import torch

ROOT_PATH = Path("/home/vasakjakub/sprout-emergence-prediction")
ANNOTATIONS_PATH = ROOT_PATH / "data" / "annotations"
SEGMENTATION_OUTPUT_DIR = ROOT_PATH / "models" / "segmentation_model" / "output"
TENSORBOARD_LOGDIR = str(SEGMENTATION_OUTPUT_DIR)

TORCH_VERSION = ".".join(torch.__version__.split(".")[:2])
CUDA_VERSION = torch.__version__.split("+")[-1]

print("Repo root:", ROOT_PATH)
print("Train annotations:", ANNOTATIONS_PATH / "Train" / "result.json")
print("Test annotations:", ANNOTATIONS_PATH / "Test" / "result.json")
print("Output dir:", SEGMENTATION_OUTPUT_DIR)
print("torch:", TORCH_VERSION, "; cuda:", CUDA_VERSION)
print("detectron2:", detectron2.__version__)

In [ ]:
from detectron2 import model_zoo
from detectron2.config import get_cfg
from detectron2.data import DatasetCatalog, MetadataCatalog
from detectron2.data.datasets import register_coco_instances
from detectron2.engine import DefaultPredictor
from detectron2.utils.logger import setup_logger
from detectron2.utils.visualizer import Visualizer

setup_logger()

## Load COCO datasets

Register train and test datasets from COCO JSON files and image folders.
Detectron2 uses these dataset names during training and evaluation.

In [ ]:
train_images = ANNOTATIONS_PATH / "Train" / "images"
train_json = ANNOTATIONS_PATH / "Train" / "result.json"

test_images = ANNOTATIONS_PATH / "Test" / "images"
test_json = ANNOTATIONS_PATH / "Test" / "result.json"

if "my_trainset" not in DatasetCatalog.list():
    register_coco_instances("my_trainset", {}, str(train_json), str(train_images))
if "my_testset" not in DatasetCatalog.list():
    register_coco_instances("my_testset", {}, str(test_json), str(test_images))

metadata = MetadataCatalog.get("my_trainset")
dataset_dicts = DatasetCatalog.get("my_trainset")
metadata_test = MetadataCatalog.get("my_testset")
dataset_dicts_test = DatasetCatalog.get("my_testset")

print(f"\n{metadata}\n \n{metadata_test}")

In [ ]:
# Check if the data is loaded correctly

for d in random.sample(dataset_dicts, 5):
    img = cv2.imread(d["file_name"])
    visualizer = Visualizer(img[:, :, ::-1], metadata=metadata, scale=0.5)
    vis = visualizer.draw_dataset_dict(d)
    plt.figure(figsize=(15, 13))
    plt.imshow(cv2.cvtColor((vis.get_image()[:, :, ::-1]), cv2.COLOR_BGR2RGB))
    plt.show()

## Train the model

Create a Mask R-CNN config, start from COCO pretrained weights, and train on the registered dataset.
Training outputs and logs are saved to `SEGMENTATION_OUTPUT_DIR`.

In [ ]:
from detectron2.engine import DefaultTrainer

cfg = get_cfg()
cfg.merge_from_file(
    model_zoo.get_config_file("COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml")
)
cfg.DATASETS.TRAIN = ("my_trainset",)
cfg.DATASETS.TEST = ()
cfg.DATALOADER.NUM_WORKERS = 4
cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url(
    "COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml"
)
cfg.SOLVER.IMS_PER_BATCH = 8
cfg.SOLVER.BASE_LR = 0.00025
cfg.SOLVER.MAX_ITER = 1300
cfg.SOLVER.STEPS = []
cfg.MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE = 512
cfg.MODEL.ROI_HEADS.NUM_CLASSES = 1

SEGMENTATION_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
cfg.OUTPUT_DIR = str(SEGMENTATION_OUTPUT_DIR)

trainer = DefaultTrainer(cfg)
trainer.resume_or_load(resume=False)
trainer.train()

## View training curves

Open TensorBoard for the training logs saved in `SEGMENTATION_OUTPUT_DIR`.

In [ ]:
%reload_ext tensorboard
%load_ext tensorboard
%tensorboard --logdir $TENSORBOARD_LOGDIR

## Evaluate the model

Load the final trained weights, run inference on the test dataset, and print COCO metrics.
The score threshold controls which predicted instances are kept.

In [ ]:
cfg.MODEL.WEIGHTS = str(SEGMENTATION_OUTPUT_DIR / "model_final.pth")
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.7
cfg.DATASETS.TEST = ("my_testset",)
predictor = DefaultPredictor(cfg)

In [ ]:
from detectron2.data import build_detection_test_loader
from detectron2.evaluation import COCOEvaluator, inference_on_dataset

test_evaluator = COCOEvaluator(
    "my_testset",
    cfg,
    distributed=False,
    output_dir=str(SEGMENTATION_OUTPUT_DIR),
)
test_loader = build_detection_test_loader(cfg, "my_testset")
print(inference_on_dataset(trainer.model, test_loader, test_evaluator))

## Visualize predictions

Show model predictions on random test images for a quick visual check.

In [ ]:
from detectron2.utils.visualizer import ColorMode

for d in random.sample(dataset_dicts_test, 11):
    im = cv2.imread(d["file_name"])
    outputs = predictor(im)
    v = Visualizer(
        im[:, :, ::-1], metadata=metadata_test, scale=0.8, instance_mode=ColorMode.IMAGE
    )
    v = v.draw_instance_predictions(outputs["instances"].to("cpu"))
    plt.figure(figsize=(15, 13))
    plt.imshow(cv2.cvtColor(v.get_image()[:, :, ::-1], cv2.COLOR_BGR2RGB))
    plt.show()